# CKPT7B: RNN + Static Features + Longer Sequences (12 months)

This notebook upgrades the RNN by:
1. **Longer sequences** (12 months of monthly counts)
2. **Static features** concatenated to the GRU output: `freq`, `monetary_total`, `latetime` (recency proxy)

Note: Using a 12‑month observation window changes the split setup compared to CKPT2 (6‑month). This is intentional to test whether longer history helps.


In [11]:
# Setup
import os, sys, random
import numpy as np
import pandas as pd

sys.path.append('..')

seed = 42
random.seed(seed)
np.random.seed(seed)


In [12]:
# Torch
import torch
import torch.nn as nn
import torch.nn.functional as F
print('Torch:', torch.__version__)


Torch: 2.11.0+cpu


In [13]:
from pathlib import Path
from src.data import load_online_retail_ii, clean_data
from src.features import create_temporal_splits_multi_extended
from src.baselines import evaluate_model


## Data + Temporal Splits (12‑month observation)
We extend the observation window to **12 months** to feed longer sequences.


In [14]:
file_2009 = Path('../data/Year 2009-2010.csv')
file_2010 = Path('../data/Year 2010-2011.csv')

if not (file_2009.exists() and file_2010.exists()):
    raise FileNotFoundError('Raw dataset files missing in ../data/')

raw = load_online_retail_ii(file_2009, file_2010)
clean = clean_data(raw)

# 12-month observation window
obs_months = 12
horizon_months = 3

# With 12-month history, earliest cutoff should be 2010-12-01 (dataset starts 2009-12)
train_cutoffs = ['2010-12-01','2011-03-01']
val_cutoff = '2011-06-01'
test_cutoff = '2011-09-01'

train_df, val_df, test_df = create_temporal_splits_multi_extended(
    clean, train_cutoffs, val_cutoff, test_cutoff,
    obs_months=obs_months, horizon_months=horizon_months
)

print('Train/Val/Test:', train_df.shape, val_df.shape, test_df.shape)


Loading Year 2009-2010...
  Loaded 525,461 rows
Loading Year 2010-2011...
  Loaded 541,910 rows

Total rows: 1,067,371

DATA CLEANING PIPELINE
Initial rows: 1,067,371
✓ Removed missing CustomerID: 243,007 rows removed
✓ Removed cancellations: 18,744 rows removed
✓ Removed invalid Quantity/Price: 71 rows removed
✓ Converted InvoiceDate to datetime
✓ Removed duplicates: 26,124 rows removed
Final rows: 779,425 (73.0% retained)
Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00
Unique customers: 5,878
Unique invoices: 36,969


CREATING TEMPORAL SPLITS (MULTI-CUTOFF TRAIN, EXTENDED)

[1/3] Training Set (Multiple Cutoffs)

--- Train cutoff: 2010-12-01 ---

Creating window (extended):
  Observation: 2009-12-01 to 2010-12-01 (12 months)
  Horizon: 2010-12-01 to 2011-03-01 (3 months)
  Observation transactions: 386,732
  Horizon transactions: 66,364
  Customers in observation: 4,266
  Final customers with features: 4,266
  Customers with target > 0: 1,411 (33.1%)

--- Train cutoff: 2011-03-

## Build Monthly Count Sequences (12 months)
We build a 12‑month count sequence per customer for the observation window.


In [15]:
monthly = clean.copy()
monthly['month'] = monthly['InvoiceDate'].dt.to_period('M').dt.to_timestamp()
monthly_counts = monthly.groupby(['CustomerID', 'month']).size().unstack(fill_value=0)
monthly_counts.columns = pd.to_datetime(monthly_counts.columns)

seq_cols = [f'm{i}' for i in range(obs_months, 0, -1)]

def build_seq_features(df_split):
    seq_rows = []
    for _, row in df_split[['CustomerID', 'cutoff_date']].iterrows():
        cid = row['CustomerID']
        cutoff = pd.to_datetime(row['cutoff_date'])
        months = pd.date_range(cutoff - pd.DateOffset(months=obs_months), periods=obs_months, freq='MS')
        if cid in monthly_counts.index:
            counts = monthly_counts.loc[cid].reindex(months, fill_value=0).values
        else:
            counts = [0] * obs_months
        seq_rows.append(counts)
    return pd.DataFrame(seq_rows, columns=seq_cols, index=df_split.index)

X_train_seq = build_seq_features(train_df)
X_val_seq = build_seq_features(val_df)
X_test_seq = build_seq_features(test_df)

print('Sequence shape:', X_train_seq.shape)


Sequence shape: (8542, 12)


## Static Features (freq, monetary, recency)
We take static features from the extended split. If `monetary_total` is missing, we fallback to `monetary_avg_invoice`.


In [16]:
# Choose static features
static_cols = []
if 'freq' in train_df.columns:
    static_cols.append('freq')
if 'monetary_total' in train_df.columns:
    static_cols.append('monetary_total')
elif 'monetary_avg_invoice' in train_df.columns:
    static_cols.append('monetary_avg_invoice')
if 'latetime' in train_df.columns:
    static_cols.append('latetime')  # recency proxy

if len(static_cols) < 2:
    raise ValueError(f'Not enough static features found. Available: {train_df.columns.tolist()}')

X_train_static = train_df[static_cols].copy()
X_val_static = val_df[static_cols].copy()
X_test_static = test_df[static_cols].copy()

y_train = train_df['target'].values
y_val = val_df['target'].values
y_test = test_df['target'].values

print('Static cols:', static_cols)


Static cols: ['freq', 'monetary_total', 'latetime']


## Dataset + DataLoader
We standardize static features using training mean/std.


In [17]:
from torch.utils.data import Dataset, DataLoader

# Standardize static features
stat_mean = X_train_static.mean(axis=0)
stat_std = X_train_static.std(axis=0).replace(0, 1.0)

X_train_static_n = (X_train_static - stat_mean) / stat_std
X_val_static_n = (X_val_static - stat_mean) / stat_std
X_test_static_n = (X_test_static - stat_mean) / stat_std

class SeqStaticDataset(Dataset):
    def __init__(self, X_seq, X_static, y):
        self.X_seq = torch.tensor(X_seq.values, dtype=torch.float32)
        self.X_static = torch.tensor(X_static.values, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.X_seq)
    def __getitem__(self, idx):
        return self.X_seq[idx].unsqueeze(-1), self.X_static[idx], self.y[idx]

train_ds = SeqStaticDataset(X_train_seq, X_train_static_n, y_train)
val_ds = SeqStaticDataset(X_val_seq, X_val_static_n, y_val)
test_ds = SeqStaticDataset(X_test_seq, X_test_static_n, y_test)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=512, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=512, shuffle=False)


## GRU + Static Head
We concatenate the GRU hidden state with static features.


In [18]:
class GRUStaticRegressor(nn.Module):
    def __init__(self, hidden_size=32, static_dim=3, dropout=0.3):
        super().__init__()
        self.gru = nn.GRU(input_size=1, hidden_size=hidden_size, num_layers=1, batch_first=True, dropout=0.0)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size + static_dim, 1)
    def forward(self, x_seq, x_static):
        out, _ = self.gru(x_seq)
        last = out[:, -1, :]
        last = self.dropout(last)
        x = torch.cat([last, x_static], dim=1)
        yhat = self.fc(x)
        return F.relu(yhat).squeeze(-1)

model = GRUStaticRegressor(hidden_size=32, static_dim=len(static_cols), dropout=0.3)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.L1Loss()


## Train + Validate
We track validation MAE and keep the best model.


In [19]:
def evaluate_loader(loader):
    model.eval()
    preds = []
    trues = []
    with torch.no_grad():
        for xb, xs, yb in loader:
            yhat = model(xb, xs)
            preds.append(yhat.cpu().numpy())
            trues.append(yb.cpu().numpy())
    preds = np.concatenate(preds)
    trues = np.concatenate(trues)
    return evaluate_model(trues, preds, 'GRU_Static')

best_val = float('inf')
best_state = None
patience = 5
no_improve = 0

epochs = 30
for epoch in range(1, epochs + 1):
    model.train()
    for xb, xs, yb in train_loader:
        optimizer.zero_grad()
        yhat = model(xb, xs)
        loss = criterion(yhat, yb)
        loss.backward()
        optimizer.step()

    val_metrics = evaluate_loader(val_loader)
    val_mae = val_metrics['MAE']
    print(f"Epoch {epoch:02d} | Val MAE: {val_mae:.4f}")

    if val_mae < best_val:
        best_val = val_mae
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= patience:
            print('Early stopping')
            break

if best_state is not None:
    model.load_state_dict(best_state)


Epoch 01 | Val MAE: 0.6841
Epoch 02 | Val MAE: 0.6294
Epoch 03 | Val MAE: 0.6193
Epoch 04 | Val MAE: 0.6150
Epoch 05 | Val MAE: 0.6130
Epoch 06 | Val MAE: 0.6168
Epoch 07 | Val MAE: 0.6052
Epoch 08 | Val MAE: 0.6104
Epoch 09 | Val MAE: 0.6057
Epoch 10 | Val MAE: 0.6021
Epoch 11 | Val MAE: 0.6038
Epoch 12 | Val MAE: 0.6064
Epoch 13 | Val MAE: 0.6084
Epoch 14 | Val MAE: 0.6023
Epoch 15 | Val MAE: 0.5986
Epoch 16 | Val MAE: 0.5911
Epoch 17 | Val MAE: 0.5982
Epoch 18 | Val MAE: 0.5992
Epoch 19 | Val MAE: 0.6138
Epoch 20 | Val MAE: 0.6068
Epoch 21 | Val MAE: 0.5945
Early stopping


## Test Evaluation + Baseline Comparison

In [20]:
test_metrics = evaluate_loader(test_loader)
print('GRU+Static Test:', test_metrics)

# Compare vs CKPT2 baseline if available
baseline_path = Path('../results/ckpt2/baseline_metrics.json')
if baseline_path.exists():
    import json
    baseline = json.loads(baseline_path.read_text())
    best_name, best_mae = None, float('inf')
    for name, m in baseline.items():
        if m['MAE'] < best_mae:
            best_mae = m['MAE']
            best_name = name
    print(f"Best CKPT2 baseline: {best_name} MAE={best_mae:.6f}")
    delta = test_metrics['MAE'] - best_mae
    print(f"GRU+Static vs best baseline: {delta:+.6f}")


GRU+Static Test: {'Model': 'GRU_Static', 'MAE': 0.9009060263633728, 'RMSE': np.float64(2.223422543548442)}
Best CKPT2 baseline: results_rf_test MAE=1.106082
GRU+Static vs best baseline: -0.205176
